# Table 3 — Computational Efficiency Comparison

This notebook automatically computes the 4 columns you can't just eyeball:

- **Parameters (M)** — how many learnable numbers the model has, in millions
- **Model Size (MB)** — how much disk space those parameters take up
- **FLOPs (G)** — how many floating-point operations one forward pass takes, in billions (a rough measure of "how much math" the model does per image)
- **Inference Time (ms)** — how long one forward pass actually takes on this machine

The **Accuracy (%) column is NOT computed here** — just copy those numbers straight from **Table 1**, which you already filled in with your training results. Accuracy depends on your fine-tuned weights and dataset, not on the architecture alone, so it can't be auto-computed the way the other 4 columns can.

Set Colab's runtime to **GPU** first (`Runtime > Change runtime type > T4 GPU`) if you want inference-time numbers that match a GPU deployment; switch to CPU runtime instead if you care about CPU inference time.

In [1]:
!pip install -q thop

## 1. Model builder (same 7 models as before)

In [2]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

NUM_CLASSES = 4  # same 4-class setup as your training notebooks

def build_model(name, num_classes=NUM_CLASSES):
    if name == "AlexNet":
        m = models.alexnet(weights=None)
        m.classifier[6] = nn.Linear(4096, num_classes)
    elif name == "VGG16":
        m = models.vgg16(weights=None)
        m.classifier[6] = nn.Linear(4096, num_classes)
    elif name == "VGG19":
        m = models.vgg19(weights=None)
        m.classifier[6] = nn.Linear(4096, num_classes)
    elif name == "ResNet18":
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "ResNet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "DenseNet121":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "EfficientNet-B0":
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m.to(device)

MODEL_NAMES = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50", "DenseNet121", "EfficientNet-B0"]

# Note: weights=None here on purpose — architecture size, FLOPs, and speed
# don't depend on which weights are loaded, only on the architecture itself.
# This also means you don't need your dataset for this notebook at all.

Using device: cuda


## 2. Compute Parameters, Model Size, FLOPs, and Inference Time for each model

In [3]:
import time
from thop import profile

results = []
INPUT_SIZE = 224  # all 7 architectures accept 224x224
N_RUNS = 50       # how many forward passes to average inference time over

for name in MODEL_NAMES:
    print(f"Measuring {name}...")
    model = build_model(name)
    model.eval()

    dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE).to(device)

    # --- Parameters ---
    total_params = sum(p.numel() for p in model.parameters())
    params_M = total_params / 1e6

    # --- Model size (MB) --- assuming float32 weights (4 bytes each)
    model_size_MB = total_params * 4 / (1024 ** 2)

    # --- FLOPs (G) --- thop gives MACs; FLOPs is roughly 2x MACs
    macs, _ = profile(model, inputs=(dummy_input,), verbose=False)
    flops_G = (2 * macs) / 1e9

    # --- Inference time (ms) --- average over N_RUNS forward passes
    with torch.no_grad():
        for _ in range(10):  # warm-up, not timed
            _ = model(dummy_input)
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(N_RUNS):
            _ = model(dummy_input)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.time()
    inference_time_ms = (end - start) / N_RUNS * 1000

    results.append({
        "Model": name,
        "Parameters (M)": round(params_M, 2),
        "Model Size (MB)": round(model_size_MB, 2),
        "FLOPs (G)": round(flops_G, 2),
        "Inference Time (ms)": round(inference_time_ms, 2),
    })

    del model
    torch.cuda.empty_cache() if device.type == "cuda" else None

print("\nDone.")

Measuring AlexNet...
Measuring VGG16...
Measuring VGG19...
Measuring ResNet18...
Measuring ResNet50...
Measuring DenseNet121...
Measuring EfficientNet-B0...

Done.


## 3. Add the Accuracy (%) column from Table 1
Paste the Accuracy values you already have in Table 1 here — they're not computed in this notebook.

In [4]:
import pandas as pd

# ---- EDIT these to match your Table 1 Accuracy (%) values ----
accuracy_from_table1 = {
    "AlexNet": 72.73,
    "VGG16": 81.82,
    "VGG19": 72.73,
    "ResNet18": 81.82,
    "ResNet50": 63.64,
    "DenseNet121": 63.64,
    "EfficientNet-B0": 63.64,
}

results_df = pd.DataFrame(results).set_index("Model")
results_df["Accuracy (%)"] = [accuracy_from_table1[m] for m in results_df.index]
results_df

,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
Model,,,,,
AlexNet,57.02,217.51,1.42,2.10,72.73
VGG16,134.28,512.23,30.93,8.84,81.82
VGG19,139.59,532.48,39.26,10.51,72.73
ResNet18,11.18,42.64,3.65,2.40,81.82
ResNet50,23.52,89.71,8.26,8.53,63.64
DenseNet121,6.96,26.54,5.79,21.49,63.64
EfficientNet-B0,4.01,15.31,0.83,9.02,63.64


## 4. (Optional) Bar chart comparing model size vs accuracy

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.bar(results_df.index, results_df["Model Size (MB)"], color="steelblue", alpha=0.7, label="Model Size (MB)")
ax1.set_ylabel("Model Size (MB)")
ax1.set_xticklabels(results_df.index, rotation=30, ha="right")

ax2 = ax1.twinx()
ax2.plot(results_df.index, results_df["Accuracy (%)"], color="darkorange", marker="o", label="Accuracy (%)")
ax2.set_ylabel("Accuracy (%)")

fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
plt.title("Model Size vs Accuracy")
plt.tight_layout()
plt.show()